# Alkuspiikki

Koitin eka tensorflowlla ja noäiden samojen optimointien jälkeen sain parhaimmillaa val_acc 93,333% val_lossilla n. 0.101. Siksi näin paljo kokeiluja. Lopulta kokeilin torch backendinä ja jo eka ois päässy tavotteesee. Päätin pitää kaiken ja ylioptimoida neuroverkkoa.

In [1]:
import os

os.environ["KERAS_BACKEND"] = "torch"

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras.layers import Input, Dense, Flatten
from keras.utils import to_categorical
from keras.models import Sequential
from sklearn.preprocessing import StandardScaler, LabelEncoder

## Ladataan data

Ladataan data ja muokataan se dataframeksi, että on helpompi tulkita käsiteltävät piirteet ja kohdearvo.

In [3]:
# Datasetin lataus
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"

columns = [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width",
    "species"
]

df = pd.read_csv(url, header=None, names=columns)

In [4]:
# tarkistetaan että datasetti latautui oikein
df

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,Iris-virginica
146,6.3,2.5,5.0,1.9,Iris-virginica
147,6.5,3.0,5.2,2.0,Iris-virginica
148,6.2,3.4,5.4,2.3,Iris-virginica


## Varmistutaan ettei ole tyhjiä arvoja datassa

In [5]:
# Tarkistetaan tyhjät arvot sarakkeittain
df.isna().sum()

sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
species         0
dtype: int64

## Jaetaan data piirteisiin ja kohteeseen

Datassa on 4 piirrettä ja 1 kohde. Kohde on nimellä "species". Se helpottaa jakoa.

In [6]:
# Jaetaan kohde y:ksi ja piirteet x:ksi
x = df.drop(columns=["species"])
y = df["species"]

## Tarkastellaan kohdeluokkia

In [7]:
print(df["species"].unique())

<StringArray>
['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
Length: 3, dtype: str


## Johtopäätökset

Ei tyhjiä, ei tarvitse niitä korjata datasta. Datassa on 4 piirrettä ja 3 eri kohdeluokkaa. Kohdeluokat ovat merkkijonoja -> ne pitää muokata eka numeerisiksi, että voi one-hot enkoodata.

## Tutkitaan maksimi- ja minimiarvoja piirteittäin

Katsotaan jokaisen piirteen minimit ja maksimit, jotta voidaan päättää tarvitaanko skaalausta.

In [8]:
sepal_length_max = x["sepal_length"].max()
sepal_length_min = x["sepal_length"].min()

sepal_width_max = x["sepal_width"].max()
sepal_width_min = x["sepal_width"].min()

petal_length_max = x["petal_length"].max()
petal_length_min = x["petal_length"].min()

petal_width_max = x["petal_width"].max()
petal_width_min = x["petal_width"].min()

print(f"Verholehden pituus: min {sepal_length_min} cm, max {sepal_length_max} cm")
print(f"Verholehden leveys: min {sepal_width_min} cm, max {sepal_width_max} cm")
print(f"Terälehden pituus: min {petal_length_min} cm, max {petal_length_max} cm")
print(f"Terälehden leveys: min {petal_width_min} cm, max {petal_width_max} cm")

Verholehden pituus: min 4.3 cm, max 7.9 cm
Verholehden leveys: min 2.0 cm, max 4.4 cm
Terälehden pituus: min 1.0 cm, max 6.9 cm
Terälehden leveys: min 0.1 cm, max 2.5 cm


## Johtopäätökset

Datan piirteiden vaihteluvälit eroavat toisistaan -> kannattaa skaalata, jotta suuremmat arvot eivät vaikuta mallin oppimiseen liikaa.

## Kohdearvojen käsittely

Ensin LabelEncoderilla numeroidaan kohdearvot numeerisiksi ja sitten one-hot enkoodataan to_categorical funktiolla.

In [9]:
# Muutetaan kohdearvot numeerisiksi
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# One-hot enkoodataan kohdearvot
y_onehot = to_categorical(y_encoded, 3)

## Datan skaalaus ja sekoitus

Aluksi sekoitetaan piirteet ja kohteet pitäen ne samoissa kohdissa. Datassa nähtiin, että kohdeluokat oli järjestyksessä. Sekoituksella varmistetaan, että koulutus- ja validointidata sisältävät eri kohdeluokkia. Ennen sekoitusta huomasin heittelyä loss-käyrässä, ja sekoituksen jälkeen oppiminen stabiloitui. Sitten skaalataan. Valitaan StandardScaler skaalaajaksi. Näin koulutusdatan piirteiden keskiarvo on 0 ja keskihajonta 1.

In [10]:
# Sekoitetaan havainnot samalla indeksijärjestyksellä
indices = np.arange(len(x))
np.random.seed(0)
np.random.shuffle(indices)

x_random = x.to_numpy()[indices]
y_shuffled = y_onehot[indices]

# Määritellään koulutusosuuden koko
train_size = int(len(x_random) * 0.8)

# StandardScaler sovitetaan vain koulutusdataan
scaler = StandardScaler()
scaler.fit(x_random[:train_size])

# Standardoidaan molemmat osuudet samalla skaalaimella
x_shuffled = scaler.transform(x_random)

print("Koulutusluokat:", y_shuffled[:train_size].sum(axis=0))
print("Validointiluokat:", y_shuffled[train_size:].sum(axis=0))
print("Piirteiden shape:", x_shuffled.shape)

Koulutusluokat: [40. 39. 41.]
Validointiluokat: [10. 11.  9.]
Piirteiden shape: (150, 4)


## Tarkistetaan onnistuiko muutokset

In [11]:
# Tarkistetaan onnistuiko molemmat skaalaus ja onehot muokkaus lukemalla ekat 5
print("Onehot ekat 5:\n")
print(y_shuffled[:5])

print("\n Skaalatut piirteet ekat 5:\n")
print(x_shuffled[:5])

Onehot ekat 5:

[[0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]

 Skaalatut piirteet ekat 5:

[[-0.02827596 -0.5746965   0.77187235  1.59434127]
 [ 0.21408943 -1.9493858   0.14177247 -0.26572355]
 [-0.39182406  2.63291187 -1.34755452 -1.32861773]
 [ 1.7894645  -0.34558162  1.45925404  0.79717064]
 [-0.99773754  0.7999928  -1.29027271 -1.32861773]]


## Päätetään mallin input arvo

Vaikka tiedetään että on 4 eri piirrettä, varmistetaan se käsittelyn ja skaalauksen jälkeen vielä shape-metodilla.

In [12]:
# Tarkistetaan shape oikean inputin määrittämiseksi
x_shuffled.shape

(150, 4)

## Johtopäätökset

Enkoodaus onnistui ja 3 eri kohdeluokkaa datassa. Skaalaus meni oikein ja saamme mallille paremmat arvot syötettyä.

# Mallinnus

## Neuronien määrien kokeilu

Input pysyy kaikissa malleissa samana ja viimeisessä Dense kerroksessa on 3 neuronia kolmea kohdeluokkaa varten. Kokeillaan ensin miten malli toimii neuroneilla 16, 32, 64, 128.

## Alustetaan mallit

Alustetaan kaikki mallit omilla neuronimäärillä ja samalla optimoijalla.

In [13]:
# 16 neuronin malli
keras.utils.set_random_seed(0)
model16 = Sequential([
    Input(shape=(4,)), # 4 shapea -> 4 eri piirrettä -> input on oltava 4
    Flatten(), # tehdään yksiulotteiseksi arvot
    Dense(16, activation='relu'),
    Dense(3, activation='softmax') # 3 neuronia 3 luokkaa varten
    ])

# 32 neuronin malli
keras.utils.set_random_seed(0)
model32 = Sequential([
    Input(shape=(4,)), # 4 shapea -> 4 eri piirrettä -> input on oltava 4
    Flatten(), # tehdään yksiulotteiseksi arvot
    Dense(32, activation='relu'),
    Dense(3, activation='softmax') # 3 neuronia 3 luokkaa varten
    ])

# 64 neuronin malli
keras.utils.set_random_seed(0)
model64 = Sequential([
    Input(shape=(4,)), # 4 shapea -> 4 eri piirrettä -> input on oltava 4
    Flatten(), # tehdään yksiulotteiseksi arvot
    Dense(64, activation='relu'),
    Dense(3, activation='softmax') # 3 neuronia 3 luokkaa varten
    ])

# 128 neuronin malli
keras.utils.set_random_seed(0)
model128 = Sequential([
    Input(shape=(4,)), # 4 shapea -> 4 eri piirrettä -> input on oltava 4
    Flatten(), # tehdään yksiulotteiseksi arvot
    Dense(128, activation='relu'),
    Dense(3, activation='softmax') # 3 neuronia 3 luokkaa varten
    ])

model_list = [model16, model32, model64, model128]

In [14]:
# Alustetaan mallit samalla optimoijalla.

for model in model_list:
    model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=0.001),   ## Määritetään eksplisiittisesti learningrate selkeyden vuoksi
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

### Pidetään batch_size oletuksena ja määritetään se eksplisiittisesti selkeyden vuoksi

In [15]:
import time

# Sovitetaan mallit ja luodaan jokaiselle oma historia.

histories = []
training_times = []

## Verbose=0 piilottaa epochit, jotta ei täytä notebookkia
for model in model_list:
    keras.utils.set_random_seed(0)
    start = time.perf_counter()
    history = model.fit(x_shuffled, y_shuffled, epochs=250, batch_size=32, validation_split=0.2, verbose=0)
    histories.append(history)

    elapsed = time.perf_counter() - start
    training_times.append(elapsed)

In [16]:
results = []

## HUOM AI:lta pyydetty koodinpätkä, että saadaan data järkevästi vertailtavaan muotoon.
## Epäonnistuin ite täs liian monta kertaa, joten pyysin korjaamaan.
for neurons, history, elapsed in zip(
    [16, 32, 64, 128], histories, training_times
):
    # Etsitään pienimmän validation lossin epoch (jätin tän kun ite en tiennyt mitä tässä tarkalleen tapahtuu)
    best_idx = np.argmin(history.history["val_loss"])

    results.append({
        "Neuronit": neurons,
        "Paras epoch": best_idx + 1,
        "Training loss": history.history["loss"][best_idx],
        "Validation loss": history.history["val_loss"][best_idx],
        "Training accuracy": history.history["accuracy"][best_idx],
        "Validation accuracy": history.history["val_accuracy"][best_idx],
        "Koulutusaika (s)": elapsed
    })

results_df = pd.DataFrame(results)

display(results_df)

,Neuronit,Paras epoch,Training loss,Validation loss,Training accuracy,Validation accuracy,Koulutusaika (s)
0,16,248,0.088813,0.129331,0.975000,0.933333,5.061903
1,32,248,0.045030,0.105423,0.983333,0.933333,4.087619
2,64,248,0.036326,0.095256,0.991667,0.933333,3.941044
3,128,202,0.036956,0.093350,0.991667,0.933333,4.337515


## Suoritusajat

In [17]:
for neurons, training_time in zip([16, 32, 64, 128], training_times):
    print(f"\nKoulutusaika neuroneilla {neurons}: {training_time:.2f} s\n")


Koulutusaika neuroneilla 16: 5.06 s


Koulutusaika neuroneilla 32: 4.09 s


Koulutusaika neuroneilla 64: 3.94 s


Koulutusaika neuroneilla 128: 4.34 s



## Johtopäätökset

Suoritusaika ei merkittävästi nouse ensimmäisen kerroksen neuroneita nostamalla. Saadaan samat koulutus- ja validointitarkkuudet neuroneilla 64 ja 128 (99,17 % ja 93,33 %), mutta validation loss on hieman parempi 128 neuronilla (`0.095256` vs `0.093350`). 16 neuronin mallilla validation loss oli huonompi (`0.129331`). Kaikilla neuronimäärillä validoinnin tarkkuus pysyi 93,33 %:ssa. Otetaan siis neuronien määräksi 128 pienimmän validation lossin perusteella. Tarkkuus on koulutuspuolella jo yli speksien eli yli 95 %, mutta validoinnin tarkkuus jää vajaaksi.

## Learning raten määritys

Etsitään paras learning rate arvoilla 0.0003, 0.001 (oletus mutta verrataan) ja 0.003. Pidetään paras eli 128 neuronia oletus batch_sizellä 32.

## Mallien luonti ja hyödynnetään edellä mallinnettua 128 0.001 learning ratella

In [18]:
keras.utils.set_random_seed(0)
model128_0003 = Sequential([
    Input(shape=(4,)), # 4 shapea -> 4 eri piirrettä -> input on oltava 4
    Flatten(), # tehdään yksiulotteiseksi arvot
    Dense(128, activation='relu'),
    Dense(3, activation='softmax') # 3 neuronia 3 luokkaa varten
    ])

model128_0003.compile(optimizer=keras.optimizers.RMSprop(learning_rate=0.0003),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

keras.utils.set_random_seed(0)
model128_003 = Sequential([
    Input(shape=(4,)), # 4 shapea -> 4 eri piirrettä -> input on oltava 4
    Flatten(), # tehdään yksiulotteiseksi arvot
    Dense(128, activation='relu'),
    Dense(3, activation='softmax') # 3 neuronia 3 luokkaa varten
    ])

model128_003.compile(optimizer=keras.optimizers.RMSprop(learning_rate=0.003),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

## Mallien koulutus

In [19]:
import time

# Tallennetaan mallit, historiat ja koulutusajat learning raten mukaan
lr_models = {
    0.0003: model128_0003,
    0.003: model128_003
}

lr_histories = {
    0.001: histories[3]  # Aikaisemmin koulutettu 128 neuronin malli 0.001 ratella
}

lr_training_times = {
    0.001: training_times[3]
}

## Verbose=0 piilottaa epochit, jotta ei täytä notebookkia
for learning_rate, model in lr_models.items():
    keras.utils.set_random_seed(0)

    start = time.perf_counter()

    history = model.fit(
        x_shuffled,
        y_shuffled,
        epochs=250,
        batch_size=32,
        validation_split=0.2,
        verbose=0
    )

    elapsed = time.perf_counter() - start

    lr_histories[learning_rate] = history
    lr_training_times[learning_rate] = elapsed

In [20]:
results = []

## HUOM AI:lta pyydetty koodinpätkä, että saadaan data järkevästi vertailtavaan muotoon.
## Epäonnistuin ite täs liian monta kertaa, joten pyysin korjaamaan.
for learning_rate, history in sorted(lr_histories.items()):

    # Etsitään epoch, jolla validation loss oli pienin
    best_idx = np.argmin(history.history["val_loss"])

    results.append({
        "Learning rate": learning_rate,
        "Neuronit": 128,
        "Paras epoch": best_idx + 1,
        "Training loss": history.history["loss"][best_idx],
        "Validation loss": history.history["val_loss"][best_idx],
        "Training accuracy": history.history["accuracy"][best_idx],
        "Validation accuracy": history.history["val_accuracy"][best_idx],
        "Koulutusaika (s)": lr_training_times[learning_rate]
    })

results_df = pd.DataFrame(results)

display(results_df)

,Learning rate,Neuronit,Paras epoch,Training loss,Validation loss,Training accuracy,Validation accuracy,Koulutusaika (s)
0,0.0003,128,248,0.081081,0.117598,0.991667,0.933333,4.146278
1,0.0010,128,202,0.036956,0.093350,0.991667,0.933333,4.337515
2,0.0030,128,117,0.033812,0.059006,0.983333,0.966667,3.751308


## Johtopäätökset

Learning rate vaikutti mallin suorituskykyyn. Arvolla 0.003 validoinnin tarkkuus nousi 93,33 %:sta 96,67 %:iin verrattuna arvoon 0.001. Myös validation loss parani (`0.093350` -> `0.059006`). Paras validation loss saavutettiin jo epochilla 117. Valitaan siis learning rateksi 0.003 seuraaviin kokeiluihin, koska sillä saatiin paras validointitarkkuus ja pienin validation loss.

## Batch size -kokeilu

Kokeillaan, saataisiinko batch sizeä pienentämällä parempi tulos. Pidetään muut asetukset samoina: 128 neuronia ja learning rate 0.003.

In [21]:
import time

batch_sizes = [8, 16, 32]

batch_histories = {}
batch_training_times = {}

for batch_size in batch_sizes:
    keras.utils.set_random_seed(0)

    model = Sequential([
        Input(shape=(4,)),
        Flatten(),
        Dense(128, activation="relu"),
        Dense(3, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.RMSprop(learning_rate=0.003),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    keras.utils.set_random_seed(0)

    start = time.perf_counter()

    history = model.fit(
        x_shuffled,
        y_shuffled,
        epochs=250,
        batch_size=batch_size,
        validation_split=0.2,
        verbose=0
    )

    elapsed = time.perf_counter() - start

    batch_histories[batch_size] = history
    batch_training_times[batch_size] = elapsed

    print(f"Batch size {batch_size}: {elapsed:.2f} s")

Batch size 8: 12.81 s
Batch size 16: 7.23 s
Batch size 32: 4.00 s


In [22]:
results = []

for batch_size, history in batch_histories.items():
    best_idx = np.argmin(history.history["val_loss"])

    results.append({
        "Batch size": batch_size,
        "Paras epoch": best_idx + 1,
        "Training loss": history.history["loss"][best_idx],
        "Validation loss": history.history["val_loss"][best_idx],
        "Training accuracy": history.history["accuracy"][best_idx],
        "Validation accuracy": history.history["val_accuracy"][best_idx],
        "Koulutusaika (s)": batch_training_times[batch_size]
    })

results_df = pd.DataFrame(results)

display(results_df.round(4))

,Batch size,Paras epoch,Training loss,Validation loss,Training accuracy,Validation accuracy,Koulutusaika (s)
0,8,101,0.0325,0.0362,0.9833,0.9667,12.8145
1,16,145,0.0246,0.0485,0.9917,0.9667,7.2343
2,32,117,0.0338,0.0590,0.9833,0.9667,4.0034


## Johtopäätökset

Batch sizen laskeminen ei parantanut validation accuracyä, joka pysyi kaikilla 96,67 %:ssa. Validation loss kuitenkin parani batch sizeä pienentämällä. Batch sizellä 32 se oli 0.0590 ja batch sizellä 8 enää 0.0362. Eroa tuli siis 0.0228. Koulutusaika nousi samalla 4,06 sekunnista 12,78 sekuntiin. Valitaan batch sizeksi 8, koska sillä saatiin pienin validation loss, vaikka koulutusaika kasvoi. Malli saavuttaa nyt tehtävänannon 95 % tarkkuustavoitteen sekä koulutus- että validointidatalla.